# My Music Age — Stage 3: Enrichment & Music Age Calculation

**Goal:** Join Stage 2's cleaned listening DataFrame with a Kaggle Spotify
catalogue to recover release years, then calculate the user's Music Age,
era distribution, and era profile.

**Course concepts applied:**
- Pandas `merge()` with left join — *Module 2, L25*
- Column selection with `usecols` (memory efficiency) — *Module 2, L16*
- NumPy weighted mean — *Module 2, L18*
- Pandas `groupby()` with multiple aggregations — *Module 2, L24*
- Integer binning (decades from years) — *Module 2, L21*
- Rule-based classification (era profile) — *Module 1, conditionals*

**Inputs required in this folder:**
- `data_pipeline.py`       (from Stage 2)
- `StreamingHistory_sample.json`  (from Stage 2, or your real Spotify export)
- `catalogue/spotify_data.csv`    (the Kaggle dataset; see Section 1 below)
- `music_age.py`           (the new Stage 3 module)


## 1. Download the Spotify catalogue from Kaggle

We use Kaggle's `amitanshjoshi/spotify-1million-tracks` dataset — 1 million
songs with columns `artist_name`, `track_name`, `year`, and more.
If you haven't downloaded it yet (80 MB), run the cell below once.

The cell is safe to re-run: it only downloads if the file isn't already there.


In [1]:
from pathlib import Path

catalogue_csv = Path("catalogue/spotify_data.csv")

if not catalogue_csv.exists():
    print("Catalogue not found locally — downloading from Kaggle…")
    # Kaggle CLI must be authenticated (KAGGLE_API_TOKEN env var set).
    !kaggle datasets download amitanshjoshi/spotify-1million-tracks -p catalogue --unzip
else:
    size_mb = catalogue_csv.stat().st_size / 1_000_000
    print(f"Catalogue already present: {catalogue_csv} ({size_mb:.1f} MB)")


Catalogue already present: catalogue/spotify_data.csv (175.8 MB)


## 2. Load and clean the listening data

This is the Stage 2 pipeline in two lines — no new work, just setting up
our input for Stage 3.


In [2]:
from data_pipeline import load_listening_history, clean_listening_history

clean = clean_listening_history(load_listening_history("StreamingHistory_sample.json"))
print(f"Cleaned listening data: {clean.shape}")
clean.head(3)


  cleaned: 3,258 → 3,013 rows (dropped 245, 7.5%)
Cleaned listening data: (3013, 11)


,end_time,artist,track,ms_played,artist_key,track_key,hour,day_of_week,month,year,minutes_played
0,2025-04-23 03:13:00,Billie Eilish,What Was I Made For?,163442,billie eilish,what was i made for?,3,Wednesday,4,2025,2.724033
1,2025-04-23 05:42:00,Red Hot Chili Peppers,Under the Bridge,282272,red hot chili peppers,under the bridge,5,Wednesday,4,2025,4.704533
2,2025-04-23 12:57:00,Radiohead,Creep,174640,radiohead,creep,12,Wednesday,4,2025,2.910667


## 3. Load the catalogue

The `load_catalogue()` function reads only the 3 columns we need
(`artist_name`, `track_name`, `year`) out of the 20 in the CSV. This
keeps memory usage low — even with 1M rows, the resulting DataFrame
is under 50 MB.

It also de-duplicates the catalogue: if a song appears multiple times
(original + remaster + compilation), we keep the *earliest* release
year, which is almost always the real original.


In [3]:
from music_age import load_catalogue

catalogue = load_catalogue("catalogue/spotify_data.csv")
print(f"Catalogue: {catalogue.shape}")
catalogue.head(3)


Catalogue: (1151880, 3)


,artist_key,track_key,release_year
0,liquor giants,half a million,2000
1,lmnop,size,2000
2,dan markell,welcome to the club,2000


In [4]:
# Sanity-check: the year column should span from about the 1920s to today
print(f"Earliest release year: {catalogue['release_year'].min()}")
print(f"Latest release year:   {catalogue['release_year'].max()}")
print(f"Median:                {int(catalogue['release_year'].median())}")


Earliest release year: 2000
Latest release year:   2023
Median:                2012


## 4. Enrich listening data with release years

This is the core merge: for each play in the listening history, look up
the track in the catalogue and attach its release year.

We use a **left join** so we don't lose any listening rows even if a
track isn't found. Tracks without a release year will just be excluded
from Music Age calculations but remain visible for other analyses.

The `stats` dict tells us how good the match rate was. Aim for > 75%;
> 90% means the catalogue has great coverage of your taste.


In [ ]:
from music_age import enrich_with_catalogue

enriched, stats = enrich_with_catalogue(clean, "catalogue/spotify_data.csv")

print("Match rate statistics:")
for k, v in stats.items():
    print(f"  {k:20s} {v}")


Match rate statistics:
  total_plays          3013
  matched_plays        2574
  match_rate_plays     85.4
  unique_tracks        40
  unique_matched       27
  match_rate_tracks    67.5


In [6]:
# What do rows look like now? Note the new 'release_year' column.
enriched.head(3)


,end_time,artist,track,ms_played,artist_key,track_key,hour,day_of_week,month,year,minutes_played,release_year
0,2025-04-23 03:13:00,Billie Eilish,What Was I Made For?,163442,billie eilish,what was i made for?,3,Wednesday,4,2025,2.724033,NaN
1,2025-04-23 05:42:00,Red Hot Chili Peppers,Under the Bridge,282272,red hot chili peppers,under the bridge,5,Wednesday,4,2025,4.704533,2014.0
2,2025-04-23 12:57:00,Radiohead,Creep,174640,radiohead,creep,12,Wednesday,4,2025,2.910667,NaN


In [7]:
# Which tracks did NOT match the catalogue? These are excluded from Music Age.
unmatched = (enriched[enriched["release_year"].isna()]
             .groupby(["artist", "track"])
             .size()
             .sort_values(ascending=False)
             .head(10))
print("Tracks not found in catalogue (top 10 by play count):")
print(unmatched if len(unmatched) else "  None — all tracks matched!")


Tracks not found in catalogue (top 10 by play count):
artist              track               
Queen               Don't Stop Me Now       108
Radiohead           Creep                    61
Billie Eilish       What Was I Made For?     53
Coldplay            Fix You                  48
Oasis               Wonderwall               48
The Beatles         Here Comes the Sun       20
Fleetwood Mac       Dreams                   19
The Beatles         Let It Be                17
A.R. Rahman         Kun Faya Kun             17
The Rolling Stones  Paint It Black           13
dtype: int64


## 5. Compute Music Age

This is the headline number. We compute two versions:

| Metric              | What it measures |
|---------------------|------------------|
| **Weighted**        | How old is the music you ACTUALLY LISTEN TO? Each play counts; heavy-rotation songs dominate. |
| **Library**         | How old is your TASTE? Each unique track counts once; breadth matters more than repetition. |

If the two numbers are close, your listening habits match your library.
If *Weighted* is much lower than *Library*, you own a lot of oldies but
mostly play new stuff. And vice-versa.


In [8]:
from music_age import compute_music_age

age = compute_music_age(enriched)
for k, v in age.items():
    print(f"  {k:25s} {v}")


  music_age_weighted        9.2
  music_age_library         10.7
  mean_release_weighted     2016.8
  mean_release_library      2015.3
  reference_year            2026
  usable_plays              2574
  usable_tracks             27


In [9]:
# Big-number display, the way the final report will show it
print()
print(f"  🎧  Your Music Age (weighted): {age['music_age_weighted']} years")
print(f"  📚  Your Music Age (library):  {age['music_age_library']} years")
print()
print(f"     — based on {age['usable_plays']:,} matched plays")
print(f"        across {age['usable_tracks']} unique tracks.")



  🎧  Your Music Age (weighted): 9.2 years
  📚  Your Music Age (library):  10.7 years

     — based on 2,574 matched plays
        across 27 unique tracks.


## 6. Era distribution — which decade dominates?

Group every matched play by the decade of its release year, then see
where the listening time piles up. This is what becomes the bar chart
in the final report.


In [10]:
from music_age import era_distribution

dist = era_distribution(enriched)
dist


,decade,minutes,play_count,pct_of_total
0,2000s,909.9,286,10.8
1,2010s,3366.9,1032,40.0
2,2020s,4138.8,1256,49.2


In [11]:
# Eyeball the distribution as text bars — the actual chart will be
# built in Stage 4. This is just a sanity check.
for _, row in dist.iterrows():
    bar = "█" * int(row["pct_of_total"])
    print(f"  {row['decade']}  {bar} {row['pct_of_total']}%")


  2000s  ██████████ 10.8%
  2010s  ████████████████████████████████████████ 40.0%
  2020s  █████████████████████████████████████████████████ 49.2%


## 7. Assign an era profile label

A single descriptive string that summarises the user's era distribution.
Current rules (in `music_age.py`):

- **`{Decade} Loyalist`** — if one decade exceeds 50%
- **Decade Hopper**       — if 4+ decades each exceed 10%
- **Future Forward**      — if 2010s + 2020s combined exceed 60%
- **Vintage Soul**        — if 1960s + 1970s + 1980s combined exceed 60%
- **Balanced Listener**   — anything else

This is a deliberately simple rule-based classifier: *descriptive*,
not predictive. No ML needed.


In [12]:
from music_age import assign_era_profile

profile = assign_era_profile(dist)
print(f"  Era Profile: {profile}")


  Era Profile: Future Forward


## 8. Top track per decade

For each decade, what did the user listen to the most? This gives
"your most-played 70s song is…" type content for the final report.


In [13]:
from music_age import top_track_per_decade

top = top_track_per_decade(enriched)
top


,decade,artist,track,minutes
0,2000s,Queen,Bohemian Rhapsody,315.8
1,2010s,Arctic Monkeys,Do I Wanna Know?,1063.8
2,2020s,The Weeknd,Save Your Tears,1118.2


## 9. Next step — Stage 4

Stage 3 produces every number the final report needs:

- ✅ Music Age (weighted + library)
- ✅ Era distribution table
- ✅ Era profile label
- ✅ Top track per decade
- ✅ Match-rate stats (for transparency)

Stage 4 is the **visual hero** — a single-page matplotlib poster in
a Spotify-inspired dark theme that brings all of this together. Then
Stage 5 wraps everything in an ipywidgets interface.
